# CrewAI: Building Multi-Agent Systems

This notebook provides a comprehensive, practical guide to CrewAI — a framework for orchestrating role-playing autonomous AI agents that work together to complete complex tasks.

All examples use **GPT-4o-mini** as the language model and **text-embedding-3-small** for embedding operations.

---

## Table of Contents

1. Installation and Environment Setup
2. Core Concepts: Agent, Task, Crew
3. Agent Properties Deep Dive
4. Task Properties Deep Dive
5. Crew Properties and Process Types
6. Built-in Tools Overview
7. Implementation 1 — Research and Report Writing Crew
8. Implementation 2 — Software Development Crew
9. Implementation 3 — Customer Support Triage Crew
10. Implementation 4 — Financial Analysis Crew with Memory
11. Implementation 5 — RAG-Powered Knowledge Base Crew
12. Advanced: Custom Tools
13. Advanced: Human-in-the-Loop
14. Advanced: Async Execution and Callbacks
15. Best Practices and Common Pitfalls

---
## 1. Installation and Environment Setup

In [ ]:
# Install required packages
%pip install crewai crewai-tools openai langchain-openai python-dotenv --quiet

In [ ]:
import os
from google.colab import userdata
from crewai.memory import Memory

# Keys
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["SERPER_API_KEY"] = userdata.get("SERPER_API_KEY")

# Force correct embedding model
memory = Memory(
    embedder={
        "provider": "openai",
        "config": {"model": "text-embedding-3-small"}
    }
)

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Define the LLM used across all agents in this notebook
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.2,           # Lower temperature for more consistent, professional output
    max_tokens=2048,
    api_key=os.environ["OPENAI_API_KEY"] # Explicitly pass the API key
)


# Define the embedding model for memory and RAG features
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=os.environ["OPENAI_API_KEY"] # Explicitly pass the API key
)

print(f"LLM: {llm.model_name}")
print(f"Embeddings: {embeddings.model}")

LLM: gpt-4o-mini
Embeddings: text-embedding-3-small


---
## 2. Core Concepts: Agent, Task, Crew

CrewAI is built around three primary primitives:

| Primitive | Purpose |
|-----------|-------------------------------------------|
| **Agent** | An autonomous entity with a role, goal, and backstory. It uses tools and an LLM to reason and act. |
| **Task** | A discrete unit of work assigned to one or more agents, with a clear expected output. |
| **Crew** | An orchestrator that manages a collection of agents and tasks, defines the execution process, and aggregates results. |

The typical workflow:
1. Define your agents with specific roles.
2. Define tasks and assign them to agents.
3. Assemble a crew and run it.


---
## 3. Agent Properties Deep Dive

Below is a minimal agent followed by a fully configured agent with explanations for every property.

In [ ]:
from crewai import Agent

minimal_agent = Agent(
    role="Data Analyst",
    goal="Extract actionable insights from raw datasets.",
    backstory=(
        "You are a senior data analyst with ten years of experience "
        "working in the financial services industry."
    ),
    llm="gpt-4o-mini",
)

print("Minimal agent created:", minimal_agent.role)

Minimal agent created: Data Analyst


In [ ]:
from crewai_tools import SerperDevTool, ScrapeWebsiteTool

# --- Fully Configured Agent ---
fully_configured_agent = Agent(
    # ----------------------------------------------------------------
    # IDENTITY PROPERTIES
    # ----------------------------------------------------------------
    role="Senior Market Research Analyst",
    # role: A short label that defines who this agent is.
    # It influences how the LLM frames its reasoning and responses.

    goal=(
        "Produce accurate, evidence-based market research reports "
        "that help executives make high-confidence investment decisions."
    ),
    # goal: What this agent is trying to achieve. The LLM uses this
    # as an objective to optimize toward throughout task execution.

    backstory=(
        "You spent 15 years as a research analyst at top-tier consulting "
        "firms including McKinsey and BCG. You have deep expertise in "
        "technology, healthcare, and consumer sectors. Your reports are "
        "known for their precision and depth."
    ),
    # backstory: A rich narrative that shapes the agent's persona.
    # The more detailed and realistic, the better the output quality.

    # ----------------------------------------------------------------
    # LLM AND TOOL PROPERTIES
    # ----------------------------------------------------------------
   llm="gpt-4o-mini",
    # llm: The language model powering this agent.
    # Each agent can have a different LLM if needed.

    tools=[SerperDevTool(), ScrapeWebsiteTool()],
    # tools: A list of Tool objects the agent can invoke.
    # Tools give agents the ability to take actions beyond text generation.

    # ----------------------------------------------------------------
    # BEHAVIOR PROPERTIES
    # ----------------------------------------------------------------
    verbose=True,
    # verbose: When True, prints the agent's internal reasoning chain
    # (Thought -> Action -> Observation loop) to stdout. Useful for debugging.

    allow_delegation=False,
    # allow_delegation: When True, this agent can delegate subtasks
    # to other agents in the crew. Set to False to keep it focused.

    max_iter=10,
    # max_iter: Maximum number of reasoning iterations before the agent
    # is forced to produce a final answer. Prevents infinite loops.

    max_rpm=20,
    # max_rpm: Maximum API requests per minute. Useful for rate-limit management.

    memory=True,
    # memory: When True, the agent retains context across task interactions
    # within the same crew run. Powered by the embedding model.

    # ----------------------------------------------------------------
    # OPTIONAL: CUSTOM SYSTEM PROMPT
    # ----------------------------------------------------------------
    # system_template: Override the default system prompt entirely.
    # Use {role}, {goal}, {backstory} as placeholders.
    # system_template="You are {role}. Your goal: {goal}. Background: {backstory}.",
)

print("Fully configured agent created:", fully_configured_agent.role)

Fully configured agent created: Senior Market Research Analyst


### Agent Property Summary

| Property | Type | Required | Description |
|---|---|---|---|
| `role` | str | Yes | The agent's job title / persona label |
| `goal` | str | Yes | What the agent aims to accomplish |
| `backstory` | str | Yes | Rich narrative that shapes behavior |
| `llm` | LLM | No | Language model (defaults to OpenAI GPT-4) |
| `tools` | list | No | Tool objects the agent can call |
| `verbose` | bool | No | Print reasoning steps (default: False) |
| `allow_delegation` | bool | No | Allow task hand-off to other agents |
| `max_iter` | int | No | Max reasoning loops (default: 15) |
| `max_rpm` | int | No | Rate limit for API calls |
| `memory` | bool | No | Enable cross-task memory |
| `system_template` | str | No | Custom system prompt template |

---
## 4. Task Properties Deep Dive

In [ ]:
from crewai import Task
from pydantic import BaseModel
from typing import List

# Define a Pydantic model for structured output
class MarketReport(BaseModel):
    company_name: str
    market_size_usd_billion: float
    key_competitors: List[str]
    growth_rate_percent: float
    recommendation: str

# --- Minimal Task ---
minimal_task = Task(
    description="Summarize the current state of the electric vehicle market.",
    expected_output="A 300-word summary covering market size, key players, and growth trends.",
    agent=minimal_agent,
)

# --- Fully Configured Task ---
research_task = Task(
    # ----------------------------------------------------------------
    # CORE PROPERTIES
    # ----------------------------------------------------------------
    description=(
        "Conduct a comprehensive market analysis for {company_name} "
        "operating in the {industry} sector. "
        "Focus on: (1) total addressable market size, "
        "(2) top 5 competitors, (3) annual growth rate, "
        "(4) a strategic recommendation for market entry."
    ),
    # description: The full instruction for the task. Supports
    # {variable} placeholders that get filled in at crew kickoff.

    expected_output=(
        "A structured JSON report containing: company_name, "
        "market_size_usd_billion, key_competitors (list of 5), "
        "growth_rate_percent, and recommendation."
    ),
    # expected_output: Describes what a successful completion looks like.
    # The LLM uses this to self-evaluate and format its response.

    agent=fully_configured_agent,
    # agent: The agent responsible for this task.

    # ----------------------------------------------------------------
    # STRUCTURED OUTPUT
    # ----------------------------------------------------------------
    output_pydantic=MarketReport,
    # output_pydantic: Force the output to conform to a Pydantic model.
    # CrewAI will validate and parse the output automatically.

    # output_json=MarketReport,  # Alternative: output as a plain dict
    # output_file="report.md",   # Alternative: save output to a file

    # ----------------------------------------------------------------
    # DEPENDENCY PROPERTIES
    # ----------------------------------------------------------------
    # context=[another_task],
    # context: A list of tasks whose outputs are passed as context
    # to this task. Enables data flow between sequential steps.

    # ----------------------------------------------------------------
    # CALLBACK
    # ----------------------------------------------------------------
    # callback=my_callback_function,
    # callback: A Python function called with the task output
    # after the task completes. Useful for logging or downstream actions.

    # ----------------------------------------------------------------
    # HUMAN INPUT
    # ----------------------------------------------------------------
    human_input=False,
    # human_input: When True, the agent pauses and requests human
    # review/approval before finalizing the output.
)

print("Tasks created successfully.")

Tasks created successfully.


### Task Property Summary

| Property | Type | Required | Description |
|---|---|---|---|
| `description` | str | Yes | What needs to be done (supports `{placeholders}`) |
| `expected_output` | str | Yes | What a completed result looks like |
| `agent` | Agent | Yes | The responsible agent |
| `context` | list[Task] | No | Upstream tasks whose output feeds this one |
| `output_pydantic` | BaseModel | No | Validate output against a Pydantic model |
| `output_json` | BaseModel | No | Return output as a plain dict |
| `output_file` | str | No | Write output to a file path |
| `callback` | callable | No | Function called on task completion |
| `human_input` | bool | No | Pause for human review before finalizing |

---
## 5. Crew Properties and Process Types

In [ ]:
from crewai import Crew, Process

# Process.sequential  — tasks run one after another in order
# Process.hierarchical — a manager agent orchestrates which agent does what

# --- Sequential Crew (most common) ---
sequential_crew = Crew(
    agents=[minimal_agent],
    tasks=[minimal_task],

    process=Process.sequential,
    # process: Defines the execution strategy.
    # Sequential: task[0] -> task[1] -> task[2] ...
    # Hierarchical: a manager agent delegates to worker agents.

    verbose=True,
    # verbose: Print crew-level orchestration logs.

    memory=False,
    # memory: Enable short-term, long-term, and entity memory
    # across the entire crew. Requires an embedding model.

    embedder={
        "provider": "openai",
        "config": {
            "model": "text-embedding-3-small"
        }
    },
    # embedder: Configuration for the embedding model used in memory.

    max_rpm=30,
    # max_rpm: Crew-level rate limit (applies across all agents).

    # share_crew=False,
    # share_crew: If True, shares crew metadata with CrewAI
    # for performance benchmarking (opt-in).

    # step_callback=my_step_fn,
    # step_callback: Called after every agent action step.

    # task_callback=my_task_fn,
    # task_callback: Called after every task completes.
)

print("Sequential crew assembled.")

Sequential crew assembled.


In [ ]:
from crewai import Agent, Task, Crew, Process

# ✅ FIX: no trailing comma
llm = "gpt-4o-mini"

worker_agent_1 = Agent(
    role="Web Researcher",
    goal="Find accurate information from online sources.",
    backstory="You are a meticulous researcher who always cites sources.",
    llm=llm,
    verbose=True,
)

worker_agent_2 = Agent(
    role="Content Writer",
    goal="Produce clear, concise, and well-structured written content.",
    backstory="You are a professional business writer with an economics background.",
    llm=llm,
    verbose=True,
)

write_task = Task(
    description="Write a 500-word executive summary on renewable energy trends.",
    expected_output="A professional executive summary suitable for a board presentation.",
    agent=worker_agent_2,
)

hierarchical_crew = Crew(
    agents=[worker_agent_1, worker_agent_2],
    tasks=[write_task],
    process=Process.hierarchical,
    manager_llm=llm,
    verbose=True,
)

print("Hierarchical crew assembled.")

Hierarchical crew assembled.


---
## 6. Built-in Tools Overview

CrewAI ships with a broad library of tools through the `crewai-tools` package.

In [ ]:
# Overview of key tools available in crewai-tools

tool_catalog = {
    "SerperDevTool": "Google Search via Serper API — good for finding current web results",
    "ScrapeWebsiteTool": "Scrape full text content from any URL",
    "FileReadTool": "Read content from a local file",
    "FileWriterTool": "Write content to a local file",
    "DirectoryReadTool": "List and read files in a directory",
    "PDFSearchTool": "Semantic search over PDF documents using RAG",
    "CSVSearchTool": "Semantic search over CSV files",
    "TXTSearchTool": "Semantic search over plain text files",
    "JSONSearchTool": "Semantic search over JSON documents",
    "DOCXSearchTool": "Semantic search over Word documents",
    "YoutubeChannelSearchTool": "Search a YouTube channel's transcripts",
    "YoutubeVideoSearchTool": "Search a YouTube video's transcript",
    "GithubSearchTool": "Search GitHub repositories and code",
    "CodeDocsSearchTool": "Semantic search over code documentation",
    "WebsiteSearchTool": "RAG-based semantic search over a website",
    "BrowserbaseLoadTool": "Load pages using a headless browser (for JS-heavy sites)",
    "DallETool": "Generate images using DALL-E",
    "VisionTool": "Analyze images with GPT-4 Vision",
}

print(f"{'Tool Name':<30} {'Description'}")
print("-" * 90)
for name, desc in tool_catalog.items():
    print(f"{name:<30} {desc}")

Tool Name                      Description
------------------------------------------------------------------------------------------
SerperDevTool                  Google Search via Serper API — good for finding current web results
ScrapeWebsiteTool              Scrape full text content from any URL
FileReadTool                   Read content from a local file
FileWriterTool                 Write content to a local file
DirectoryReadTool              List and read files in a directory
PDFSearchTool                  Semantic search over PDF documents using RAG
CSVSearchTool                  Semantic search over CSV files
TXTSearchTool                  Semantic search over plain text files
JSONSearchTool                 Semantic search over JSON documents
DOCXSearchTool                 Semantic search over Word documents
YoutubeChannelSearchTool       Search a YouTube channel's transcripts
YoutubeVideoSearchTool         Search a YouTube video's transcript
GithubSearchTool               

In [ ]:
from crewai.memory import Memory

custom_memory = Memory(
    embedder={
        "provider": "openai",
        "config": {
            "model": "text-embedding-3-small"
        }
    }
)

In [ ]:
from crewai import Agent, Task, Crew, Process
from crewai_tools import SerperDevTool, ScrapeWebsiteTool

# ✅ Use string OR CrewAI LLM (recommended simple way)
llm = "gpt-4o-mini"

# --- Agents ---
researcher = Agent(
    role="Senior Investment Research Analyst",
    goal=(
        "Gather comprehensive, factual information about companies "
        "including financial performance, competitive position, and market trends."
    ),
    backstory=(
        "You are a CFA charterholder with 12 years of experience in equity research "
        "at a leading investment bank. You rely only on verifiable data and cite all sources."
    ),
    llm=llm,
    tools=[SerperDevTool(), ScrapeWebsiteTool()],
    verbose=True,
    allow_delegation=False,
    max_iter=8,
    memory=True,
)

report_writer = Agent(
    role="Business Report Writer",
    goal=(
        "Convert research into clear, structured, executive-ready investment memos."
    ),
    backstory=(
        "You are a former Bloomberg and Wall Street Journal writer. "
        "You follow the Pyramid Principle and write with clarity and precision."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
    max_iter=6,
    memory=True,
)

# --- Tasks ---
research_task = Task(
    description=(
        "Research {company_name} in the {industry} industry.\n\n"
        "Collect:\n"
        "1. Latest revenue and profit figures\n"
        "2. Top 3 competitors with positioning\n"
        "3. Recent news impacting stock price\n"
        "4. Analyst consensus rating\n\n"
        "IMPORTANT: Include source URLs for every fact."
    ),
    expected_output=(
        "Structured bullet-point research brief with clearly labeled sections "
        "and source links for each data point."
    ),
    agent=researcher,
)

writing_task = Task(
    description=(
        "Using the research brief, write a one-page investment memo on {company_name}.\n\n"
        "Structure:\n"
        "- Executive Summary (2 sentences)\n"
        "- Company Overview (1 paragraph)\n"
        "- Financial Highlights (bullets)\n"
        "- Competitive Landscape (1 paragraph)\n"
        "- Risks (3 bullets)\n"
        "- Recommendation (1 paragraph)\n\n"
        "Ensure clarity, conciseness, and professional tone."
    ),
    expected_output=(
        "A polished, executive-ready investment memo in Markdown format."
    ),
    agent=report_writer,
    context=[research_task],
    output_file="investment_memo.md",
)

# --- Crew ---
research_crew = Crew(
    agents=[researcher, report_writer],
    tasks=[research_task, writing_task],
    process=Process.sequential,
    memory=custom_memory,
    verbose=True,
)

# --- Run ---
result = await research_crew.kickoff_async(inputs={
    "company_name": "NVIDIA",
    "industry": "semiconductors and AI computing"
})

print("\n" + "=" * 60)
print("FINAL OUTPUT")
print("=" * 60)
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 98bfa4df-02fb-4825-9efe-e038d7edce60                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research NVIDIA in the semiconductors and AI computing industry.                                         │
│                                                                                                                 │
│  Collect:                                                                                                       │
│  1. Latest revenue and profit figures                                                                           │
│  2. Top 3 competitors with positioning                                                                          │
│  3. Recent news impacting stock price                                                                           │
│  4. Analyst consensus rating                                                                                    │
│                                                                                                                 │
│  IMPORTANT: Include source URLs for every fact.                                                                 │
│  ID: c49563f8-2830-49b6-8830-d1280899b980                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 🧠 Memory Retrieval ──────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Retrieval Started                                                                                       │
│  Status: Retrieving...                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 🧠 Memory Retrieved ──────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Retrieval Completed                                                                                     │
│  Time: 2350.11ms                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Investment Research Analyst                                                                      │
│                                                                                                                 │
│  Task: Research NVIDIA in the semiconductors and AI computing industry.                                         │
│                                                                                                                 │
│  Collect:                                                                                                       │
│  1. Latest revenue and profit figures                                                                           │
│  2. Top 3 competitors with positioning                                                                          │
│  3. Recent news impacting stock price                                                                           │
│  4. Analyst consensus rating                                                                                    │
│                                                                                                                 │
│  IMPORTANT: Include source URLs for every fact.                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Business Report Writer                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```markdown                                                                                                    │
│  # Investment Memo: NVIDIA                                                                                      │
│                                                                                                                 │
│  ## Executive Summary                                                                                           │
│  NVIDIA stands at the forefront of the AI and graphics processing industries, evidenced by a staggering 101%    │
│  year-over-year revenue growth reported in Q2 2023. With a strong competitive position and favorable analyst    │
│  sentiment, the company is poised for continued success.                                                        │
│                                                                                                                 │
│  ## Company Overview                                                                                            │
│  NVIDIA Corporation, founded in 1993 and headquartered in Santa Clara, California, is a global leader in        │
│  artificial intelligence (AI) computing and graphics technology. The company's innovative platforms and         │
│  solutions encompass gaming, data centers, and automotive technologies. NVIDIA's GPUs are critical to modern    │
│  computing, serving as the backbone for AI developments and high-performance workloads, which are increasingly  │
│  integral to businesses in various sectors.                                                                     │
│                                                                                                                 │
│  ## Financial Highlights                                                                                        │
│  - **Q2 2023 Revenue**: $13.51 billion (101% increase YoY)                                                      │
│  - **Net Income**: $6.19 billion, or $2.48 per share (up from $656 million, or $0.26 per share YoY)             │
│  - **Market Reaction**: Stock price rose by over 20% following earnings report in August 2023                   │
│  - **Analyst Rating**: "Strong Buy" with a consensus price target of $550                                       │
│                                                                                                                 │
│  ## Competitive Landscape                                                                                       │
│  NVIDIA faces stiff competition from key players in the semiconductor and AI markets. Advanced Micro Devices    │
│  (AMD) is rapidly expanding its data center product lineup, targeting AI and machine learning applications.     │
│  Intel is repositioning itself to contend in AI through its Gaudi and Habana Labs initiatives, while Qualcomm,  │
│  known for mobile chipsets, is increasingly investing in AI and edge computing solutions. NVIDIA's leadership   │
│  in both AI and graphics provides a distinct advantage, though these competitors are making aggressive moves    │
│  to capture market share.                                                                                       │
│                                                                                                                 │
│  ## Risks                                              

╭──────────────────────────────────────────────── 🧠 Memory Save ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Save Started                                                                                            │
│  Status: Saving...                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Using the research brief, write a one-page investment memo on NVIDIA.                                    │
│                                                                                                                 │
│  Structure:                                                                                                     │
│  - Executive Summary (2 sentences)                                                                              │
│  - Company Overview (1 paragraph)                                                                               │
│  - Financial Highlights (bullets)                                                                               │
│  - Competitive Landscape (1 paragraph)                                                                          │
│  - Risks (3 bullets)                                                                                            │
│  - Recommendation (1 paragraph)                                                                                 │
│                                                                                                                 │
│  Ensure clarity, conciseness, and professional tone.                                                            │
│  Agent: Business Report Writer                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── ✅ Memory Saved ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Save Completed                                                                                          │
│  Source: Unified Memory                                                                                         │
│  Time: 3184.58ms                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FINAL OUTPUT
```markdown
# Investment Memo: NVIDIA

## Executive Summary
NVIDIA stands at the forefront of the AI and graphics processing industries, evidenced by a staggering 101% year-over-year revenue growth reported in Q2 2023. With a strong competitive position and favorable analyst sentiment, the company is poised for continued success.

## Company Overview
NVIDIA Corporation, founded in 1993 and headquartered in Santa Clara, California, is a global leader in artificial intelligence (AI) computing and graphics technology. The company's innovative platforms and solutions encompass gaming, data centers, and automotive technologies. NVIDIA's GPUs are critical to modern computing, serving as the backbone for AI developments and high-performance workloads, which are increasingly integral to businesses in various sectors.

## Financial Highlights
- **Q2 2023 Revenue**: $13.51 billion (101% increase YoY)
- **Net Income**: $6.19 billion, or $2.48 per share (up from $656 million, or $0.2

In [ ]:
from crewai import Agent, Task, Crew, Process
from crewai_tools import SerperDevTool, ScrapeWebsiteTool

llm = "gpt-4o-mini"

# --- Manager Agent ---
manager = Agent(
    role="Investment Research Manager",
    goal=(
        "Oversee the research and writing process to produce a high-quality "
        "investment memo. Decide task order and delegate effectively."
    ),
    backstory=(
        "You are a senior portfolio manager overseeing analysts and writers. "
        "You break down problems and assign tasks efficiently."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=True,   # 🔥 IMPORTANT
)

# --- Worker Agents ---
researcher = Agent(
    role="Senior Investment Research Analyst",
    goal="Gather comprehensive and factual company data with sources.",
    backstory="CFA with 12 years of equity research experience.",
    llm=llm,
    tools=[SerperDevTool(), ScrapeWebsiteTool()],
    verbose=True,
    allow_delegation=False,
)

report_writer = Agent(
    role="Business Report Writer",
    goal="Write clear, structured investment memos.",
    backstory="Former Bloomberg/WSJ writer.",
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

# --- Tasks (no strict ordering now) ---
research_task = Task(
    description="Research {company_name} in {industry} with financials, competitors, news, ratings. Include sources.",
    expected_output="Bullet-point research brief with sources.",
    agent=researcher,
)

writing_task = Task(
    description="Write a structured investment memo using research.",
    expected_output="Executive-ready memo in Markdown.",
    agent=report_writer,
)

# --- Crew ---
research_crew = Crew(
    agents=[researcher, report_writer],   # ✅ ONLY workers
    tasks=[research_task, writing_task],
    process=Process.hierarchical,
    manager_agent=manager,                # ✅ manager here
    memory=custom_memory,
    verbose=True,
)
# --- Run ---
result = await research_crew.kickoff_async(inputs={
    "company_name": "NVIDIA",
    "industry": "semiconductors and AI computing"
})

print("\n" + "=" * 60)
print("FINAL OUTPUT")
print("=" * 60)
print(result)



╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                                            │
│                                                                              │
│  Your preference has been saved. Future Crew/Flow executions will not        │
│  collect traces.                                                             │
│                                                                              │
│  To enable tracing later, do any one of these:                               │
│  • Set tracing=True in your Crew/Flow code                                   │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file               │
│  • Run: crewai traces enable                                                 │
│                                                                              │
╰─────────────────────────

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 1ee5ff73-37dc-413b-b97a-5f171a88f46d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research NVIDIA in semiconductors and AI computing with financials, competitors, news, ratings. Include  │
│  sources.                                                                                                       │
│  ID: 5f563fc4-7e0b-4332-8680-2094a6d2865e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 🧠 Memory Retrieval ──────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Retrieval Started                                                                                       │
│  Status: Retrieving...                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ❌ Memory Query Error ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Query Failed                                                                                            │
│  Source: Unified Memory                                                                                         │
│  Error: Memory requires an embedder for vector search but initialization failed:                                │
│  OpenAIEmbeddingFunction.__init__() got an unexpected keyword argument 'model'                                  │
│                                                                                                                 │
│  To fix this, do one of the following:                                                                          │
│    - Set OPENAI_API_KEY for the default embedder (text-embedding-3-large)                                       │
│    - Pass a different embedder: Memory(embedder={{"provider": "google", "config": {{...}}}})                    │
│    - Pass a callable: Memory(embedder=my_embedding_function)                                                    │
│                                                                                                                 │
│  Docs: https://docs.crewai.com/concepts/memory                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[CrewAIEventsBus] Warning: Ending event 'memory_save_failed' emitted with empty scope stack. Missing starting 
event?

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Write a structured investment memo using research.                                                       │
│  Agent: Investment Research Manager                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ❌ Memory Save Error ──────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Save Failed                                                                                             │
│  Source: Unified Memory                                                                                         │
│  Error: Memory requires an embedder for vector search but initialization failed:                                │
│  OpenAIEmbeddingFunction.__init__() got an unexpected keyword argument 'model'                                  │
│                                                                                                                 │
│  To fix this, do one of the following:                                                                          │
│    - Set OPENAI_API_KEY for the default embedder (text-embedding-3-large)                                       │
│    - Pass a different embedder: Memory(embedder={{"provider": "google", "config": {{...}}}})                    │
│    - Pass a callable: Memory(embedder=my_embedding_function)                                                    │
│                                                                                                                 │
│  Docs: https://docs.crewai.com/concepts/memory                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FINAL OUTPUT
# Investment Memo: NVIDIA Corporation 

## Executive Summary
NVIDIA Corporation, a leader in the semiconductor industry, is at the forefront of advancements in artificial intelligence (AI) and computing technologies. With strong financial performance driven by its business segments—gaming, data center, and automotive—NVIDIA is well-positioned for future growth. This memo evaluates NVIDIA’s financial status, competitive landscape, recent developments, and analyst ratings. 

## Company Overview
- **Company Name**: NVIDIA Corporation
- **Founded**: 1993
- **Headquarters**: Santa Clara, California
- **Industry**: Semiconductors, AI Computing

## Financial Performance
For Fiscal Year 2023, NVIDIA reported notable financial metrics:
- **Revenue**: Estimated at $26 billion.
- **Net Income**: Approximately $9.8 billion.
- **Earnings per Share (EPS)**: $3.08, reflecting growth from previous years.
- **Market Capitalization**: Over $1 trillion, making NVIDIA one of the most valuabl

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 1ee5ff73-37dc-413b-b97a-5f171a88f46d                                                                       │
│  Final Output: # Investment Memo: NVIDIA Corporation                                                            │
│                                                                                                                 │
│  ## Executive Summary                                                                                           │
│  NVIDIA Corporation, a leader in the semiconductor industry, is at the forefront of advancements in artificial  │
│  intelligence (AI) and computing technologies. With strong financial performance driven by its business         │
│  segments—gaming, data center, and automotive—NVIDIA is well-positioned for future growth. This memo evaluates  │
│  NVIDIA’s financial status, competitive landscape, recent developments, and analyst ratings.                    │
│                                                                                                                 │
│  ## Company Overview                                                                                            │
│  - **Company Name**: NVIDIA Corporation                                                                         │
│  - **Founded**: 1993                                                                                            │
│  - **Headquarters**: Santa Clara, California                                                                    │
│  - **Industry**: Semiconductors, AI Computing                                                                   │
│                                                                                                                 │
│  ## Financial Performance                                                                                       │
│  For Fiscal Year 2023, NVIDIA reported notable financial metrics:                                               │
│  - **Revenue**: Estimated at $26 billion.                                                                       │
│  - **Net Income**: Approximately $9.8 billion.                                                                  │
│  - **Earnings per Share (EPS)**: $3.08, reflecting growth from previous years.                                  │
│  - **Market Capitalization**: Over $1 trillion, making NVIDIA one of the most valuable companies in the         │
│  semiconductor sector.                                                                                          │
│                                                                                                                 │
│  ## Business Segments                                                                                           │
│  1. **Gaming**:                                                                                                 │
│     - Largest revenue driver, benefiting from the increasing demand for gaming technologies and GPU sales.      │
│                                                                                                                 │
│  2. **Data Center**:                                                                                            │
│     - Exhibiting rapid growth fueled by demand for AI, machine learning, and cloud computing services.          │
│                                                                                                                 │
│  3. **Automotive**:                                   

In [ ]:
from crewai import Agent, Task, Crew, Process
from crewai_tools import SerperDevTool, ScrapeWebsiteTool

llm = "gpt-4o-mini"

# =========================
# 🧠 PLANNER AGENT (NEW 🔥)
# =========================
planner = Agent(
    role="Execution Planner",
    goal=(
        "Create a step-by-step execution plan for producing an investment memo. "
        "Clearly define which agent should do what and in what order."
    ),
    backstory="Expert in breaking down complex workflows into clear execution steps.",
    llm=llm,
    verbose=True,
)

# =========================
# 🧠 MANAGER
# =========================
manager = Agent(
    role="Chief Investment Officer",
    goal=(
        "Execute the plan efficiently by delegating tasks to the right agents. "
        "Ensure high-quality output."
    ),
    backstory="Senior decision-maker managing analysts and writers.",
    llm=llm,
    verbose=True,
    allow_delegation=True,
)

# =========================
# 👨‍💻 WORKERS
# =========================
researcher = Agent(
    role="Equity Research Analyst",
    goal="Gather accurate financial and market data with sources.",
    backstory="CFA with deep research expertise.",
    llm=llm,
    tools=[SerperDevTool(), ScrapeWebsiteTool()],
    verbose=True,
)

validator = Agent(
    role="Data Validator",
    goal="Verify accuracy and consistency of research.",
    backstory="Audit specialist.",
    llm=llm,
    verbose=True,
)

writer = Agent(
    role="Investment Memo Writer",
    goal="Write structured and professional investment memo.",
    backstory="Financial journalist.",
    llm=llm,
    verbose=True,
)

reviewer = Agent(
    role="Senior Reviewer",
    goal="Refine and improve clarity and quality.",
    backstory="Portfolio manager.",
    llm=llm,
    verbose=True,
)

# =========================
# 📋 STEP 1: PLAN TASK
# =========================
planning_task = Task(
    description=(
        "Create a detailed execution plan to analyze {company_name} in {industry}.\n"
        "Include:\n"
        "- Step-by-step workflow\n"
        "- Which agent performs each step\n"
        "- Expected outputs\n"
        "Keep it structured and clear."
    ),
    expected_output="Step-by-step execution plan.",
    agent=planner,
)

# =========================
# 📋 STEP 2: EXECUTION TASKS
# =========================
research_task = Task(
    description="Perform research on {company_name} with financials, competitors, news.",
    expected_output="Research with sources.",
    agent=researcher,
)

validation_task = Task(
    description="Validate research for accuracy and completeness.",
    expected_output="Validated research.",
    agent=validator,
)

writing_task = Task(
    description="Write investment memo.",
    expected_output="Structured memo.",
    agent=writer,
)

review_task = Task(
    description="Refine and finalize memo.",
    expected_output="Final polished memo.",
    agent=reviewer,
)

# =========================
# 🧩 CREW 1 → PLANNING
# =========================
planning_crew = Crew(
    agents=[planner],
    tasks=[planning_task],
    process=Process.sequential,
    verbose=True,
)

# =========================
# ▶️ RUN PLANNING FIRST
# =========================
plan = await planning_crew.kickoff_async(inputs={
    "company_name": "NVIDIA",
    "industry": "semiconductors and AI computing"
})

print("\n" + "=" * 60)
print("🧠 EXECUTION PLAN (BEFORE RUNNING AGENTS)")
print("=" * 60)
print(plan)

# =========================
# 🧩 CREW 2 → AUTONOMOUS EXECUTION
# =========================
execution_crew = Crew(
    agents=[researcher, validator, writer, reviewer],
    tasks=[research_task, validation_task, writing_task, review_task],
    process=Process.hierarchical,
    manager_agent=manager,
    verbose=True,
)

# =========================
# ▶️ RUN EXECUTION
# =========================
result = await execution_crew.kickoff_async(inputs={
    "company_name": "NVIDIA",
    "industry": "semiconductors and AI computing"
})

print("\n" + "=" * 60)
print("🚀 FINAL OUTPUT")
print("=" * 60)
print(result)



╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                                            │
│                                                                              │
│  Your preference has been saved. Future Crew/Flow executions will not        │
│  collect traces.                                                             │
│                                                                              │
│  To enable tracing later, do any one of these:                               │
│  • Set tracing=True in your Crew/Flow code                                   │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file               │
│  • Run: crewai traces enable                                                 │
│                                                                              │
╰─────────────────────────

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 170b2b2a-09d1-4487-8ca7-ac40d8929f73                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Create a detailed execution plan to analyze NVIDIA in semiconductors and AI computing.                   │
│  Include:                                                                                                       │
│  - Step-by-step workflow                                                                                        │
│  - Which agent performs each step                                                                               │
│  - Expected outputs                                                                                             │
│  Keep it structured and clear.                                                                                  │
│  ID: a312cd4c-be6b-4185-b738-1b1bc4ea93cf                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Execution Planner                                                                                       │
│                                                                                                                 │
│  Task: Create a detailed execution plan to analyze NVIDIA in semiconductors and AI computing.                   │
│  Include:                                                                                                       │
│  - Step-by-step workflow                                                                                        │
│  - Which agent performs each step                                                                               │
│  - Expected outputs                                                                                             │
│  Keep it structured and clear.                                                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


...

╭─────────────────────────────────────── ✅ Tool Execution Completed (#4) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Output: **Investment Memo: NVIDIA Corporation (NVDA)**                                                         │
│  **Date:** Q3 2023                                                                                              │
│  **Prepared by:** [Your Name]                                                                                   │
│  **Position:** Senior Reviewer, Portfolio Manager                                                               │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  **Executive Summary**                                                                                          │
│                                                                                                                 │
│  This memo provides an in-depth analysis of NVIDIA Corporation (NVDA) as of Q3 2023, encompassing its           │
│  financial performance, competitive landscape, recent developments, and strategic insights. The insights        │
│  herein are critical for discerning NVIDIA’s investment potential in the rapidly evolving technology sector,    │
│  particularly in graphics processing, artificial intelligence, and gaming industries.                           │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  **1. Financial Performance Overview**                                                                          │
│                                                                                                                 │
│  As of Q3 2023, NVIDIA has demonstrated robust financial performance driven by substantial growth in its        │
│  Gaming and Data Center segments. Key metrics include:                                                          │
│                                                                                                                 │
│  - **Revenue:** NVIDIA reported a revenue of $6.9 billion for Q3 2023, marking a year-over-year increase of     │
│  56%. This growth is primarily attributed to its dominance in the gaming and AI sectors.                        │
│                                                                                                                 │
│  - **Gross Margin:** NVIDIA maintained a gross margin of 67.3%, reflecting its ability to capitalize on         │
│  high-demand products while controlling costs effectively.                                                      │
│                                                                                                                 │
│  - **Net Income:** The net income for the quarter rose to $2.5 billion, underscoring profitability              │
│  enhancements driven by operational efficiencies and increased sales volume.                                    │
│                                                                                                                 │
│  ---                                                   

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Chief Investment Officer                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Investment Memo: NVIDIA Corporation (NVDA)**                                                                 │
│  **Date:** Q3 2023                                                                                              │
│  **Prepared by:** [Your Name]                                                                                   │
│  **Position:** Senior Reviewer, Portfolio Manager                                                               │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  **Executive Summary**                                                                                          │
│                                                                                                                 │
│  This memo provides an in-depth analysis of NVIDIA Corporation (NVDA) as of Q3 2023, encompassing its           │
│  financial performance, competitive landscape, recent developments, and strategic insights. The insights        │
│  herein are critical for discerning NVIDIA’s investment potential in the rapidly evolving technology sector,    │
│  particularly in graphics processing, artificial intelligence, and gaming industries.                           │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  **1. Financial Performance Overview**                                                                          │
│                                                                                                                 │
│  As of Q3 2023, NVIDIA has demonstrated robust financial performance driven by substantial growth in its        │
│  Gaming and Data Center segments. Key metrics include:                                                          │
│                                                                                                                 │
│  - **Revenue:** NVIDIA reported a revenue of **$26.0 billion**, marking a year-over-year increase of            │
│  **70.68%**. This growth is primarily attributed to its dominance in the gaming and AI sectors.                 │
│                                                                                                                 │
│  - **Gross Margin:** NVIDIA maintained a gross margin of **74.15%**, reflecting its ability to capitalize on    │
│  high-demand products while controlling costs effectively.                                                      │
│                                                                                                                 │
│  - **Net Income:** The net income for the quarter rose to approximately **$16 billion**, underscoring           │
│  profitability enhancements driven by operational efficiencies and increased sales volume.                      │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Refine and finalize memo.                                                                                │
│  Agent: Chief Investment Officer                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


🚀 FINAL OUTPUT
**Investment Memo: NVIDIA Corporation (NVDA)**  
**Date:** Q3 2023  
**Prepared by:** [Your Name]  
**Position:** Senior Reviewer, Portfolio Manager  

---

**Executive Summary**  

This memo provides an in-depth analysis of NVIDIA Corporation (NVDA) as of Q3 2023, encompassing its financial performance, competitive landscape, recent developments, and strategic insights. The insights herein are critical for discerning NVIDIA’s investment potential in the rapidly evolving technology sector, particularly in graphics processing, artificial intelligence, and gaming industries.

---

**1. Financial Performance Overview**  

As of Q3 2023, NVIDIA has demonstrated robust financial performance driven by substantial growth in its Gaming and Data Center segments. Key metrics include:

- **Revenue:** NVIDIA reported a revenue of **$26.0 billion**, marking a year-over-year increase of **70.68%**. This growth is primarily attributed to its dominance in the gaming and AI sectors.
  


---
## 7. Implementation 1 — Research and Report Writing Crew

**Scenario:** An investment firm needs a quick market research report on a company before a decision meeting. Two agents collaborate: one researches, one writes.

---
## 8. Implementation 2 — Software Development Crew

**Scenario:** A startup needs a Python module built. A product manager, software engineer, and QA reviewer collaborate to produce production-ready code.

In [ ]:
from crewai import Agent, Task, Crew, Process

# --- Agents ---
product_manager = Agent(
    role="Technical Product Manager",
    goal="Translate business requirements into precise technical specifications.",
    backstory=(
        "You have 8 years of experience as a PM at companies like Stripe and Twilio. "
        "You write specs that engineers can implement without asking follow-up questions."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

software_engineer = Agent(
    role="Senior Python Engineer",
    goal="Write clean, well-documented, production-quality Python code.",
    backstory=(
        "You are a principal engineer who has worked on backend systems processing "
        "millions of requests per day. You follow PEP 8, write docstrings, and always "
        "include type hints and error handling."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

qa_reviewer = Agent(
    role="QA Engineer",
    goal="Identify bugs, edge cases, and security issues in code before it ships.",
    backstory=(
        "You are a QA lead who has caught critical bugs that saved companies from "
        "production incidents. You are thorough and never approve code without "
        "checking for input validation, error handling, and at least 3 edge cases."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

# --- Tasks ---
spec_task = Task(
    description=(
        "Write a technical specification for a Python function called "
        "`calculate_compound_interest`. "
        "Requirements: accepts principal (float), annual rate (float), "
        "years (int), and compounds_per_year (int, default 12). "
        "Returns final amount (float) rounded to 2 decimal places. "
        "Must handle invalid inputs gracefully."
    ),
    expected_output=(
        "A technical spec document including: function signature, parameter "
        "descriptions, return type, formula to use, and 3 input/output examples."
    ),
    agent=product_manager,
)

coding_task = Task(
    description=(
        "Implement the function described in the specification. "
        "Use type hints, a comprehensive docstring, raise ValueError for invalid inputs, "
        "and include 5 unit tests using pytest within the same file."
    ),
    expected_output=(
        "A complete, runnable Python file with the function implementation "
        "and pytest test suite."
    ),
    agent=software_engineer,
    context=[spec_task],
)

review_task = Task(
    description=(
        "Review the provided Python code for: (1) correctness of the compound interest "
        "formula, (2) completeness of input validation, (3) test coverage gaps, "
        "(4) any security or performance concerns. "
        "Provide a PASS or FAIL verdict with specific line-by-line comments."
    ),
    expected_output=(
        "A code review report with: verdict (PASS/FAIL), a list of issues found "
        "(if any), and the corrected code if changes are needed."
    ),
    agent=qa_reviewer,
    context=[coding_task],
    output_file="code_review_report.md",
)

# --- Crew ---
dev_crew = Crew(
    agents=[product_manager, software_engineer, qa_reviewer],
    tasks=[spec_task, coding_task, review_task],
    process=Process.sequential,
    verbose=True,
)

result = await dev_crew.kickoff_async()

print("\n" + "=" * 60)
print("CODE REVIEW RESULT")
print("=" * 60)
print(result)

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 7181f895-d7d4-426f-8142-418d8aaac35d                                                                       │
│  Final Output: **Investment Memo: NVIDIA Corporation (NVDA)**                                                   │
│  **Date:** Q3 2023                                                                                              │
│  **Prepared by:** [Your Name]                                                                                   │
│  **Position:** Senior Reviewer, Portfolio Manager                                                               │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  **Executive Summary**                                                                                          │
│                                                                                                                 │
│  This memo provides an in-depth analysis of NVIDIA Corporation (NVDA) as of Q3 2023, encompassing its           │
│  financial performance, competitive landscape, recent developments, and strategic insights. The insights        │
│  herein are critical for discerning NVIDIA’s investment potential in the rapidly evolving technology sector,    │
│  particularly in graphics processing, artificial intelligence, and gaming industries.                           │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  **1. Financial Performance Overview**                                                                          │
│                                                                                                                 │
│  As of Q3 2023, NVIDIA has demonstrated robust financial performance driven by substantial growth in its        │
│  Gaming and Data Center segments. Key metrics include:                                                          │
│                                                                                                                 │
│  - **Revenue:** NVIDIA reported a revenue of **$26.0 billion**, marking a year-over-year increase of            │
│  **70.68%**. This growth is primarily attributed to its dominance in the gaming and AI sectors.                 │
│                                                                                                                 │
│  - **Gross Margin:** NVIDIA maintained a gross margin of **74.15%**, reflecting its ability to capitalize on    │
│  high-demand products while controlling costs effectively.                                                      │
│                                                                                                                 │
│  - **Net Income:** The net income for the quarter rose to approximately **$16 billion**, underscoring           │
│  profitability enhancements driven by operational efficiencies and increased sales volume.                      │
│                                                       

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 66b9f0f6-5168-470b-87ff-da33f8e010e7                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Write a technical specification for a Python function called `calculate_compound_interest`.              │
│  Requirements: accepts principal (float), annual rate (float), years (int), and compounds_per_year (int,        │
│  default 12). Returns final amount (float) rounded to 2 decimal places. Must handle invalid inputs gracefully.  │
│  ID: 659e5ae2-953e-49a9-8579-3bc960fcd88d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Technical Product Manager                                                                               │
│                                                                                                                 │
│  Task: Write a technical specification for a Python function called `calculate_compound_interest`.              │
│  Requirements: accepts principal (float), annual rate (float), years (int), and compounds_per_year (int,        │
│  default 12). Returns final amount (float) rounded to 2 decimal places. Must handle invalid inputs gracefully.  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: QA Engineer                                                                                             │
│                                                                                                                 │
│  Task: Review the provided Python code for: (1) correctness of the compound interest formula, (2) completeness  │
│  of input validation, (3) test coverage gaps, (4) any security or performance concerns. Provide a PASS or FAIL  │
│  verdict with specific line-by-line comments.                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: QA Engineer                                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ## Code Review Report                                                                                          │
│                                                                                                                 │
│  **Verdict: FAIL**                                                                                              │
│                                                                                                                 │
│  ### Issues Found:                                                                                              │
│                                                                                                                 │
│  1. **Correctness of the Compound Interest Formula**:                                                           │
│     - The formula implementation is correct according to the parameters defined in the docstring.               │
│                                                                                                                 │
│  2. **Completeness of Input Validation**:                                                                       │
│     - The input validations for the parameters are mostly correct. However, it could be improved by handling    │
│  cases where inputs are inappropriate types (e.g., strings or None).                                            │
│                                                                                                                 │
│  3. **Test Coverage Gaps**:                                                                                     │
│     - There should be additional test cases for edge cases:                                                     │
│       - A test case for zero years to see if it returns the principal unaltered.                                │
│       - A test case for very large values to verify behavior with extreme numbers and potential overflow.       │
│       - A test case for non-integer years and compounds per year to verify type handling.                       │
│                                                                                                                 │
│  4. **Security and Performance Concerns**:                                                                      │
│     - The function does not explicitly handle type safety, which can lead to runtime errors if unexpected       │
│  types are provided.                                                                                            │
│     - There is no handling of very large or invalid floats, which could potentially cause performance issues    │
│  or unexpected behavior.                                                                                        │
│                                                                                                                 │
│  ### Suggested Corrected Code:                                                                                  │
│                                                                                                                 │
│  Here’s the modified code with improvements incorporated:                                                       │
│                                                                                                                 │
│  ```python                                             

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Review the provided Python code for: (1) correctness of the compound interest formula, (2) completeness  │
│  of input validation, (3) test coverage gaps, (4) any security or performance concerns. Provide a PASS or FAIL  │
│  verdict with specific line-by-line comments.                                                                   │
│  Agent: QA Engineer                                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


CODE REVIEW RESULT
## Code Review Report

**Verdict: FAIL**

### Issues Found:

1. **Correctness of the Compound Interest Formula**:
   - The formula implementation is correct according to the parameters defined in the docstring.

2. **Completeness of Input Validation**:
   - The input validations for the parameters are mostly correct. However, it could be improved by handling cases where inputs are inappropriate types (e.g., strings or None).

3. **Test Coverage Gaps**:
   - There should be additional test cases for edge cases:
     - A test case for zero years to see if it returns the principal unaltered.
     - A test case for very large values to verify behavior with extreme numbers and potential overflow.
     - A test case for non-integer years and compounds per year to verify type handling.

4. **Security and Performance Concerns**:
   - The function does not explicitly handle type safety, which can lead to runtime errors if unexpected types are provided.
   - There is no handl

---
## 9. Implementation 3 — Customer Support Triage Crew

**Scenario:** An e-commerce company receives customer complaints. Agents classify tickets, draft responses, and escalate high-priority issues.

In [ ]:
from crewai import Agent, Task, Crew, Process
from pydantic import BaseModel
from typing import Literal

class TriageResult(BaseModel):
    ticket_id: str
    category: Literal["billing", "shipping", "product", "technical", "other"]
    priority: Literal["low", "medium", "high", "critical"]
    sentiment: Literal["positive", "neutral", "negative", "angry"]
    requires_escalation: bool
    summary: str

# --- Agents ---
triage_agent = Agent(
    role="Customer Support Triage Specialist",
    goal=(
        "Accurately classify customer tickets by category, priority, and sentiment "
        "so that the right team handles them with the right urgency."
    ),
    backstory=(
        "You managed the support inbox at a high-growth SaaS company for 5 years. "
        "You can instantly recognize whether a ticket is a billing dispute, a technical "
        "bug, or a shipping complaint, and you know exactly what makes a ticket critical."
    ),
    llm=llm,
    verbose=True,
)

response_agent = Agent(
    role="Senior Customer Success Manager",
    goal=(
        "Write empathetic, professional, and solution-focused responses that "
        "resolve customer issues on the first contact whenever possible."
    ),
    backstory=(
        "You have a background in psychology and customer experience. You hold a "
        "CSAT score of 4.9/5.0 across 10,000 tickets. Your responses always "
        "acknowledge the issue, apologize where appropriate, and provide a clear next step."
    ),
    llm=llm,
    verbose=True,
)

# --- Tasks ---
triage_task = Task(
    description=(
        "Analyze the following customer support ticket and classify it:\n\n"
        "Ticket ID: {ticket_id}\n"
        "Customer Message: {customer_message}\n\n"
        "Return structured output with: category, priority (critical if the customer "
        "mentions a chargeback, legal action, or data breach), sentiment, "
        "requires_escalation (True if critical or angry + high), and a 1-sentence summary."
    ),
    expected_output="A structured JSON object matching the TriageResult schema.",
    agent=triage_agent,
    output_pydantic=TriageResult,
)

response_task = Task(
    description=(
        "Using the triage classification, draft a customer-facing response email. "
        "Tone must match sentiment: warmer and more apologetic for negative/angry tickets. "
        "If requires_escalation is True, mention that a senior specialist will follow up "
        "within 2 business hours. Always include a ticket reference number."
    ),
    expected_output=(
        "A complete customer response email with subject line and body. "
        "Professional, empathetic, and under 200 words."
    ),
    agent=response_agent,
    context=[triage_task],
)

# --- Crew ---
support_crew = Crew(
    agents=[triage_agent, response_agent],
    tasks=[triage_task, response_task],
    process=Process.sequential,
    verbose=True,
    tracing=True,
)

# Simulate a customer complaint
result = await support_crew.kickoff_async(inputs={
    "ticket_id": "TKT-20481",
    "customer_message": (
        "I ordered a laptop 3 weeks ago and it still has not arrived. "
        "The tracking link on your website has not updated in 10 days. "
        "This is completely unacceptable. I need this for work and if I don't "
        "receive it by tomorrow I will dispute the charge with my credit card company."
    )
})

print("\n" + "=" * 60)
print("SUPPORT CREW OUTPUT")
print("=" * 60)
print(result)

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 66b9f0f6-5168-470b-87ff-da33f8e010e7                                                                       │
│  Final Output: ## Code Review Report                                                                            │
│                                                                                                                 │
│  **Verdict: FAIL**                                                                                              │
│                                                                                                                 │
│  ### Issues Found:                                                                                              │
│                                                                                                                 │
│  1. **Correctness of the Compound Interest Formula**:                                                           │
│     - The formula implementation is correct according to the parameters defined in the docstring.               │
│                                                                                                                 │
│  2. **Completeness of Input Validation**:                                                                       │
│     - The input validations for the parameters are mostly correct. However, it could be improved by handling    │
│  cases where inputs are inappropriate types (e.g., strings or None).                                            │
│                                                                                                                 │
│  3. **Test Coverage Gaps**:                                                                                     │
│     - There should be additional test cases for edge cases:                                                     │
│       - A test case for zero years to see if it returns the principal unaltered.                                │
│       - A test case for very large values to verify behavior with extreme numbers and potential overflow.       │
│       - A test case for non-integer years and compounds per year to verify type handling.                       │
│                                                                                                                 │
│  4. **Security and Performance Concerns**:                                                                      │
│     - The function does not explicitly handle type safety, which can lead to runtime errors if unexpected       │
│  types are provided.                                                                                            │
│     - There is no handling of very large or invalid floats, which could potentially cause performance issues    │
│  or unexpected behavior.                                                                                        │
│                                                                                                                 │
│  ### Suggested Corrected Code:                                                                                  │
│                                                                                                                 │
│  Here’s the modified code with improvements incorporated:                                                       │
│                                                                                                                 │
│  ```python                                            

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 68dc17cf-a105-4142-96d0-83a7331458ed                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyze the following customer support ticket and classify it:                                           │
│                                                                                                                 │
│  Ticket ID: TKT-20481                                                                                           │
│  Customer Message: I ordered a laptop 3 weeks ago and it still has not arrived. The tracking link on your       │
│  website has not updated in 10 days. This is completely unacceptable. I need this for work and if I don't       │
│  receive it by tomorrow I will dispute the charge with my credit card company.                                  │
│                                                                                                                 │
│  Return structured output with: category, priority (critical if the customer mentions a chargeback, legal       │
│  action, or data breach), sentiment, requires_escalation (True if critical or angry + high), and a 1-sentence   │
│  summary.                                                                                                       │
│  ID: 9e0b71fb-539e-4cd3-b636-ecf8cad71fde                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Customer Support Triage Specialist                                                                      │
│                                                                                                                 │
│  Task: Analyze the following customer support ticket and classify it:                                           │
│                                                                                                                 │
│  Ticket ID: TKT-20481                                                                                           │
│  Customer Message: I ordered a laptop 3 weeks ago and it still has not arrived. The tracking link on your       │
│  website has not updated in 10 days. This is completely unacceptable. I need this for work and if I don't       │
│  receive it by tomorrow I will dispute the charge with my credit card company.                                  │
│                                                                                                                 │
│  Return structured output with: category, priority (critical if the customer mentions a chargeback, legal       │
│  action, or data breach), sentiment, requires_escalation (True if critical or angry + high), and a 1-sentence   │
│  summary.                                                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Customer Success Manager                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Subject: Urgent Assistance Regarding Your Laptop Order (Ticket TKT-20481)                                      │
│                                                                                                                 │
│  Dear [Customer's Name],                                                                                        │
│                                                                                                                 │
│  I sincerely apologize for the frustration and inconvenience you've experienced regarding the delayed arrival   │
│  of your laptop order. I completely understand how important this is for you, and I’m sorry for any stress      │
│  this situation has caused.                                                                                     │
│                                                                                                                 │
│  To address your concern promptly, I have escalated your ticket to one of our senior specialists, who will      │
│  follow up with you within the next 2 business hours. We are committed to resolving this issue swiftly,         │
│  ensuring your order arrives as soon as possible.                                                               │
│                                                                                                                 │
│  Thank you for your patience as we work towards a solution. Your satisfaction is very important to us, and we   │
│  appreciate you bringing this matter to our attention.                                                          │
│                                                                                                                 │
│  Best regards,                                                                                                  │
│                                                                                                                 │
│  [Your Name]                                                                                                    │
│  Senior Customer Success Manager                                                                                │
│  [Your Company]                                                                                                 │
│  Ticket Reference: TKT-20481                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Using the triage classification, draft a customer-facing response email. Tone must match sentiment:      │
│  warmer and more apologetic for negative/angry tickets. If requires_escalation is True, mention that a senior   │
│  specialist will follow up within 2 business hours. Always include a ticket reference number.                   │
│  Agent: Senior Customer Success Manager                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


SUPPORT CREW OUTPUT
Subject: Urgent Assistance Regarding Your Laptop Order (Ticket TKT-20481)

Dear [Customer's Name],

I sincerely apologize for the frustration and inconvenience you've experienced regarding the delayed arrival of your laptop order. I completely understand how important this is for you, and I’m sorry for any stress this situation has caused.

To address your concern promptly, I have escalated your ticket to one of our senior specialists, who will follow up with you within the next 2 business hours. We are committed to resolving this issue swiftly, ensuring your order arrives as soon as possible.

Thank you for your patience as we work towards a solution. Your satisfaction is very important to us, and we appreciate you bringing this matter to our attention.

Best regards,

[Your Name]  
Senior Customer Success Manager  
[Your Company]  
Ticket Reference: TKT-20481


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 68dc17cf-a105-4142-96d0-83a7331458ed                                                                       │
│  Final Output: Subject: Urgent Assistance Regarding Your Laptop Order (Ticket TKT-20481)                        │
│                                                                                                                 │
│  Dear [Customer's Name],                                                                                        │
│                                                                                                                 │
│  I sincerely apologize for the frustration and inconvenience you've experienced regarding the delayed arrival   │
│  of your laptop order. I completely understand how important this is for you, and I’m sorry for any stress      │
│  this situation has caused.                                                                                     │
│                                                                                                                 │
│  To address your concern promptly, I have escalated your ticket to one of our senior specialists, who will      │
│  follow up with you within the next 2 business hours. We are committed to resolving this issue swiftly,         │
│  ensuring your order arrives as soon as possible.                                                               │
│                                                                                                                 │
│  Thank you for your patience as we work towards a solution. Your satisfaction is very important to us, and we   │
│  appreciate you bringing this matter to our attention.                                                          │
│                                                                                                                 │
│  Best regards,                                                                                                  │
│                                                                                                                 │
│  [Your Name]                                                                                                    │
│  Senior Customer Success Manager                                                                                │
│  [Your Company]                                                                                                 │
│  Ticket Reference: TKT-20481                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---
## 10. Implementation 4 — Financial Analysis Crew with Memory

**Scenario:** A multi-turn financial analysis session where agents remember previous findings and build on them across multiple crew runs.

In [ ]:
from crewai import Agent, Task, Crew, Process

# Memory-enabled crew persists knowledge between runs
# using short-term (in-run) and long-term (vector store) memory.

financial_analyst = Agent(
    role="Quantitative Financial Analyst",
    goal=(
        "Analyze financial metrics, identify trends, and compute key ratios "
        "to assess the financial health of companies."
    ),
    backstory=(
        "You hold a PhD in Financial Economics from Wharton and spent 10 years "
        "at a quant hedge fund. You think in numbers, ratios, and distributions."
    ),
    llm=llm,
    memory=True,
    verbose=True,
)

strategy_advisor = Agent(
    role="Corporate Strategy Advisor",
    goal=(
        "Translate financial analysis into actionable strategic recommendations "
        "that leadership teams can present to their boards."
    ),
    backstory=(
        "You are a former managing director at a Big Four consulting firm. "
        "You bridge the gap between financial data and strategic action."
    ),
    llm=llm,
    memory=True,
    verbose=True,
)

# Provide raw financial data as context in the task description
financial_data = """
Company: Acme Corp
FY2023 Revenue: $4.2B (up 18% YoY)
Gross Margin: 62%
Operating Income: $840M
Net Income: $610M
EPS: $3.82
Debt/Equity: 0.45
Current Ratio: 2.1
Free Cash Flow: $520M
R&D Spend: 14% of revenue
Employee Count: 18,400 (up 6% YoY)
"""

analysis_task = Task(
    description=(
        f"Analyze the following financial data for Acme Corp:\n{financial_data}\n"
        "Compute: P/E ratio context, revenue growth sustainability assessment, "
        "capital efficiency score, and flag any red flags or strengths. "
        "Compare ratios to industry medians for enterprise software companies."
    ),
    expected_output=(
        "A financial analysis report with computed ratios, trend commentary, "
        "3 key strengths, and 2 areas of concern."
    ),
    agent=financial_analyst,
)

strategy_task = Task(
    description=(
        "Based on the financial analysis, provide 3 strategic recommendations "
        "for Acme Corp's leadership team. Each recommendation must include: "
        "the strategic action, rationale grounded in the financial data, "
        "expected outcome, and a risk if not acted upon."
    ),
    expected_output=(
        "Three structured strategic recommendations formatted as a board-ready slide outline."
    ),
    agent=strategy_advisor,
    context=[analysis_task],
)

# Memory-enabled crew — uses text-embedding-3-small for vector memory
financial_crew = Crew(
    agents=[financial_analyst, strategy_advisor],
    tasks=[analysis_task, strategy_task],
    process=Process.sequential,
    memory=True,
    embedder={
        "provider": "openai",
        "config": {"model": "text-embedding-3-small"}
    },
    verbose=True,
)

result = await financial_crew.kickoff_async()
print(result)



╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                                            │
│                                                                              │
│  Your preference has been saved. Future Crew/Flow executions will not        │
│  collect traces.                                                             │
│                                                                              │
│  To enable tracing later, do any one of these:                               │
│  • Set tracing=True in your Crew/Flow code                                   │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file               │
│  • Run: crewai traces enable                                                 │
│                                                                              │
╰─────────────────────────

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 0a491c98-4d84-4fd4-9406-495dec4214bd                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyze the following financial data for Acme Corp:                                                      │
│                                                                                                                 │
│  Company: Acme Corp                                                                                             │
│  FY2023 Revenue: $4.2B (up 18% YoY)                                                                             │
│  Gross Margin: 62%                                                                                              │
│  Operating Income: $840M                                                                                        │
│  Net Income: $610M                                                                                              │
│  EPS: $3.82                                                                                                     │
│  Debt/Equity: 0.45                                                                                              │
│  Current Ratio: 2.1                                                                                             │
│  Free Cash Flow: $520M                                                                                          │
│  R&D Spend: 14% of revenue                                                                                      │
│  Employee Count: 18,400 (up 6% YoY)                                                                             │
│                                                                                                                 │
│  Compute: P/E ratio context, revenue growth sustainability assessment, capital efficiency score, and flag any   │
│  red flags or strengths. Compare ratios to industry medians for enterprise software companies.                  │
│  ID: 1f8b18b0-6463-4db8-a60e-32378f9fc6b1                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 🧠 Memory Retrieved ──────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Retrieval Completed                                                                                     │
│  Time: 2260.48ms                                                                                                │
│  Content:                                                                                                       │
│  Relevant memories:                                                                                             │
│  - (score=0.69) NVIDIA's net income for Q2 2023 was $6.19 billion, or $2.48 per share, compared to $656         │
│  million (or $0.26 per share) from the same quarter last year.                                                  │
│    categories: financial, company performance                                                                   │
│    entities: ['NVIDIA']                                                                                         │
│    dates: ['Q2 2023', 'last year']                                                                              │
│    topics: ['net income', 'financial performance', 'quarterly earnings']                                        │
│  - (score=0.68) NVIDIA reported a revenue of $13.51 billion for the fiscal second quarter ended July 30, 2023,  │
│  marking an increase of 101% year-over-year.                                                                    │
│    categories: finance, technology, company performance                                                         │
│    entities: ['NVIDIA']                                                                                         │
│    dates: ['July 30, 2023']                                                                                     │
│    topics: ['revenue', 'fiscal performance', 'year-over-year growth']                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Quantitative Financial Analyst                                                                          │
│                                                                                                                 │
│  Task: Analyze the following financial data for Acme Corp:                                                      │
│                                                                                                                 │
│  Company: Acme Corp                                                                                             │
│  FY2023 Revenue: $4.2B (up 18% YoY)                                                                             │
│  Gross Margin: 62%                                                                                              │
│  Operating Income: $840M                                                                                        │
│  Net Income: $610M                                                                                              │
│  EPS: $3.82                                                                                                     │
│  Debt/Equity: 0.45                                                                                              │
│  Current Ratio: 2.1                                                                                             │
│  Free Cash Flow: $520M                                                                                          │
│  R&D Spend: 14% of revenue                                                                                      │
│  Employee Count: 18,400 (up 6% YoY)                                                                             │
│                                                                                                                 │
│  Compute: P/E ratio context, revenue growth sustainability assessment, capital efficiency score, and flag any   │
│  red flags or strengths. Compare ratios to industry medians for enterprise software companies.                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Corporate Strategy Advisor                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Strategic Recommendations for Acme Corp's Leadership Team                                                    │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## Recommendation 1: Enhance Employee Productivity and Training Programs                                       │
│                                                                                                                 │
│  **Strategic Action:**                                                                                          │
│  Implement a robust employee productivity improvement program that includes comprehensive training,             │
│  mentorship, and performance metrics evaluations.                                                               │
│                                                                                                                 │
│  **Rationale:**                                                                                                 │
│  Despite a 6% increase in employee count, there is a need to correlate this growth with improved productivity   │
│  to mitigate the risk of inflating operating costs. The current operating income stands at $840M, indicating a  │
│  strong potential to derive higher output from the workforce.                                                   │
│                                                                                                                 │
│  **Expected Outcome:**                                                                                          │
│  Enhanced employee productivity will lead to improved operational efficiency and potentially increase net       │
│  income, above the current $610M, thus maximizing returns on the current investment in human capital.           │
│                                                                                                                 │
│  **Risk if Not Acted Upon:**                                                                                    │
│  Failure to enhance employee productivity may lead to increased operational costs without corresponding         │
│  revenue growth. This could dilute profit margins and negatively impact overall business performance,           │
│  endangering the company's competitive standing.                                                                │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## Recommendation 2: Increase Investment in R&D to Sustain Revenue Growth                                      │
│                                                                                                                 │
│  **Strategic Action:**                                                                                          │
│  Allocate a higher percentage of revenue to R&D, increa

╭──────────────────────────────────────────────── 🧠 Memory Save ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Save Started                                                                                            │
│  Status: Saving...                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Based on the financial analysis, provide 3 strategic recommendations for Acme Corp's leadership team.    │
│  Each recommendation must include: the strategic action, rationale grounded in the financial data, expected     │
│  outcome, and a risk if not acted upon.                                                                         │
│  Agent: Corporate Strategy Advisor                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── ✅ Memory Saved ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Save Completed                                                                                          │
│  Source: Unified Memory                                                                                         │
│  Time: 3438.41ms                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

# Strategic Recommendations for Acme Corp's Leadership Team

---

## Recommendation 1: Enhance Employee Productivity and Training Programs

**Strategic Action:**
Implement a robust employee productivity improvement program that includes comprehensive training, mentorship, and performance metrics evaluations.

**Rationale:**
Despite a 6% increase in employee count, there is a need to correlate this growth with improved productivity to mitigate the risk of inflating operating costs. The current operating income stands at $840M, indicating a strong potential to derive higher output from the workforce.

**Expected Outcome:**
Enhanced employee productivity will lead to improved operational efficiency and potentially increase net income, above the current $610M, thus maximizing returns on the current investment in human capital.

**Risk if Not Acted Upon:**
Failure to enhance employee productivity may lead to increased operational costs without corresponding revenue growth. This could dilute

---
## 11. Implementation 5 — RAG-Powered Knowledge Base Crew

**Scenario:** A legal team wants agents to answer questions grounded in their internal policy documents using Retrieval-Augmented Generation (RAG).

In [ ]:
from crewai import Agent, Task, Crew, Process
from crewai_tools import PDFSearchTool, TXTSearchTool

from google.colab import files

uploaded = files.upload()

# PDFSearchTool uses text-embedding-3-small under the hood
# to chunk, embed, and semantically retrieve relevant passages
# from the provided document before passing them to the agent.

# Assumes you have a PDF file at this path
policy_tool = PDFSearchTool(
    pdf=list(uploaded.keys())[0],
    config={
        "llm": {
            "provider": "openai",
            "config": {"model": "gpt-4o-mini", "temperature": 0.0}
        },
        "embedder": {
            "provider": "openai",
            "config": {"model": "text-embedding-3-small"}
        }
    }
)

legal_researcher = Agent(
    role="Corporate Legal Researcher",
    goal=(
        "Answer legal and compliance questions accurately by retrieving "
        "relevant clauses from internal policy documents."
    ),
    backstory=(
        "You are a paralegal at a Fortune 500 company specializing in "
        "corporate compliance and employment law. You always cite the "
        "specific section of the policy document in your answers."
    ),
    llm=llm,
    tools=[policy_tool],
    verbose=True,
)

rag_task = Task(
    description=(
        "Answer the following employee question using only the content of "
        "the company policy document. Do not speculate beyond what the "
        "document states. Question: {employee_question}"
    ),
    expected_output=(
        "A direct answer to the question, followed by the exact policy section "
        "that supports the answer, and a note if the policy is silent on the topic."
    ),
    agent=legal_researcher,
)

rag_crew = Crew(
    agents=[legal_researcher],
    tasks=[rag_task],
    process=Process.sequential,
    verbose=True,
)

# Example query
result = await rag_crew.kickoff_async(inputs={
    "employee_question": (
        "Am I entitled to equal employment opportunities if I have been with the company "
        "for 6 months, and if so, whats the details?"
    )
})

print(result)

Saving hrpolicy.pdf to hrpolicy (8).pdf


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: ee31fb35-96f6-422d-a546-7a1479c92652                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Answer the following employee question using only the content of the company policy document. Do not     │
│  speculate beyond what the document states. Question: Am I entitled to equal employment opportunities if I      │
│  have been with the company for 6 months, and if so, whats the details?                                         │
│  ID: 6695818c-61db-4ad3-a30a-b07f8d4e2a95                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Corporate Legal Researcher                                                                              │
│                                                                                                                 │
│  Task: Answer the following employee question using only the content of the company policy document. Do not     │
│  speculate beyond what the document states. Question: Am I entitled to equal employment opportunities if I      │
│  have been with the company for 6 months, and if so, whats the details?                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#4) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_a_pdfs_content                                                                                    │
│  Args: {'query': 'equal employment opportunities'}                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Corporate Legal Researcher                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Yes, you are entitled to equal employment opportunities after being with the company for 6 months. The         │
│  company is committed to providing equal employment opportunities to all employees and applicants, with         │
│  employment decisions being based solely on qualifications, performance, and business needs. Discrimination or  │
│  harassment of any kind will not be tolerated.                                                                  │
│                                                                                                                 │
│  This is supported by the policy section:                                                                       │
│                                                                                                                 │
│  **Section 11: Equal Employment Opportunity**                                                                   │
│  "The company is committed to providing equal employment opportunities to all employees and applicants.         │
│  Employment decisions will be based solely on qualifications, performance, and business needs. Discrimination   │
│  or harassment of any kind will not be tolerated."                                                              │
│                                                                                                                 │
│  The policy does not specify any conditions related to the duration of employment (such as the 6-month period)  │
│  for the entitlement to equal employment opportunities.                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Answer the following employee question using only the content of the company policy document. Do not     │
│  speculate beyond what the document states. Question: Am I entitled to equal employment opportunities if I      │
│  have been with the company for 6 months, and if so, whats the details?                                         │
│  Agent: Corporate Legal Researcher                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Yes, you are entitled to equal employment opportunities after being with the company for 6 months. The company is committed to providing equal employment opportunities to all employees and applicants, with employment decisions being based solely on qualifications, performance, and business needs. Discrimination or harassment of any kind will not be tolerated.

This is supported by the policy section:

**Section 11: Equal Employment Opportunity**  
"The company is committed to providing equal employment opportunities to all employees and applicants. Employment decisions will be based solely on qualifications, performance, and business needs. Discrimination or harassment of any kind will not be tolerated."

The policy does not specify any conditions related to the duration of employment (such as the 6-month period) for the entitlement to equal employment opportunities.


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: ee31fb35-96f6-422d-a546-7a1479c92652                                                                       │
│  Final Output: Yes, you are entitled to equal employment opportunities after being with the company for 6       │
│  months. The company is committed to providing equal employment opportunities to all employees and applicants,  │
│  with employment decisions being based solely on qualifications, performance, and business needs.               │
│  Discrimination or harassment of any kind will not be tolerated.                                                │
│                                                                                                                 │
│  This is supported by the policy section:                                                                       │
│                                                                                                                 │
│  **Section 11: Equal Employment Opportunity**                                                                   │
│  "The company is committed to providing equal employment opportunities to all employees and applicants.         │
│  Employment decisions will be based solely on qualifications, performance, and business needs. Discrimination   │
│  or harassment of any kind will not be tolerated."                                                              │
│                                                                                                                 │
│  The policy does not specify any conditions related to the duration of employment (such as the 6-month period)  │
│  for the entitlement to equal employment opportunities.                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---
## 12. Advanced: Custom Tools

You can build any custom tool by subclassing `BaseTool` or using the `@tool` decorator.

In [ ]:
from crewai.tools import BaseTool, tool
import requests
from pydantic import Field

# --- Method 1: @tool decorator (for simple functions) ---
@tool("Currency Converter")
def currency_converter(amount: float, from_currency: str, to_currency: str) -> str:
    """
    Converts a monetary amount from one currency to another.
    Uses a live exchange rate API.
    Example: currency_converter(100, 'USD', 'EUR')
    """
    # In production, use a real API like frankfurter.app
    # This is a placeholder showing the pattern
    url = f"https://api.frankfurter.app/latest?amount={amount}&from={from_currency}&to={to_currency}"
    try:
        response = requests.get(url, timeout=5)
        data = response.json()
        converted = data["rates"][to_currency]
        return f"{amount} {from_currency} = {converted:.2f} {to_currency}"
    except Exception as e:
        return f"Error fetching exchange rate: {str(e)}"


# --- Method 2: BaseTool subclass (for complex tools with state) ---
class DatabaseLookupTool(BaseTool):
    name: str = "Customer Database Lookup"
    description: str = (
        "Looks up customer information from the internal database "
        "by customer ID. Returns account status, plan, and join date."
    )
    # Simulate an in-memory database
    database: dict = Field(default_factory=lambda: {
        "CUST-001": {"name": "Apex Industries", "plan": "Enterprise", "status": "Active", "since": "2021-03-15"},
        "CUST-002": {"name": "Nova Retail Group", "plan": "Pro", "status": "Past Due", "since": "2022-08-22"},
        "CUST-003": {"name": "Meridian Healthcare", "plan": "Starter", "status": "Active", "since": "2023-11-01"},
    })

    def _run(self, customer_id: str) -> str:
        record = self.database.get(customer_id.upper())
        if record:
            return (
                f"Customer: {record['name']} | Plan: {record['plan']} | "
                f"Status: {record['status']} | Customer since: {record['since']}"
            )
        return f"No customer found with ID: {customer_id}"


# Use the custom tools in an agent
billing_agent = Agent(
    role="Billing Support Specialist",
    goal="Resolve billing disputes by looking up account details and performing currency calculations.",
    backstory="You handle billing inquiries and account status checks for a SaaS platform.",
    llm=llm,
    tools=[DatabaseLookupTool(), currency_converter],
    verbose=True,
)

billing_task = Task(
    description=(
        "Look up customer CUST-002 and determine their account status. "
        "If their account is past due, calculate what their monthly Pro plan fee of "
        "$299 USD would be in EUR and GBP for their international billing team."
    ),
    expected_output=(
        "Account status summary with currency conversions and a recommended action."
    ),
    agent=billing_agent,
)

billing_crew = Crew(
    agents=[billing_agent],
    tasks=[billing_task],
    process=Process.sequential,
    verbose=True,
)

result = await billing_crew.kickoff_async()
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 0e01ff99-2984-41f2-b7bd-af0d4818eb56                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Look up customer CUST-002 and determine their account status. If their account is past due, calculate    │
│  what their monthly Pro plan fee of $299 USD would be in EUR and GBP for their international billing team.      │
│  ID: 9c086b94-edb9-4c19-a085-49745f80bf46                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Billing Support Specialist                                                                              │
│                                                                                                                 │
│  Task: Look up customer CUST-002 and determine their account status. If their account is past due, calculate    │
│  what their monthly Pro plan fee of $299 USD would be in EUR and GBP for their international billing team.      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool customer_database_lookup executed with result: Customer: Nova Retail Group | Plan: Pro | Status: Past Due | Customer since: 2022-08-22...


╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: customer_database_lookup                                                                                 │
│  Args: {'customer_id': 'CUST-002'}                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Billing Support Specialist                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Account Status Summary for Nova Retail Group**                                                               │
│                                                                                                                 │
│  - **Plan:** Pro                                                                                                │
│  - **Account Status:** Past Due                                                                                 │
│  - **Customer Since:** August 22, 2022                                                                          │
│                                                                                                                 │
│  **Monthly Pro Plan Fee:**                                                                                      │
│  - In EUR: €261.68                                                                                              │
│  - In GBP: £222.14                                                                                              │
│                                                                                                                 │
│  **Recommended Action:** It is advised to reach out to Nova Retail Group to resolve the past due account        │
│  status and confirm the payment of the monthly Pro plan fee of €261.68 or £222.14, depending on their           │
│  preferred billing currency.                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Look up customer CUST-002 and determine their account status. If their account is past due, calculate    │
│  what their monthly Pro plan fee of $299 USD would be in EUR and GBP for their international billing team.      │
│  Agent: Billing Support Specialist                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

/tmp/ipykernel_1221/973388822.py:78: RuntimeWarning: coroutine 'Crew.kickoff_async' was never awaited
  result = await billing_crew.kickoff_async()


**Account Status Summary for Nova Retail Group**

- **Plan:** Pro
- **Account Status:** Past Due
- **Customer Since:** August 22, 2022

**Monthly Pro Plan Fee:**
- In EUR: €261.68
- In GBP: £222.14

**Recommended Action:** It is advised to reach out to Nova Retail Group to resolve the past due account status and confirm the payment of the monthly Pro plan fee of €261.68 or £222.14, depending on their preferred billing currency.


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 0e01ff99-2984-41f2-b7bd-af0d4818eb56                                                                       │
│  Final Output: **Account Status Summary for Nova Retail Group**                                                 │
│                                                                                                                 │
│  - **Plan:** Pro                                                                                                │
│  - **Account Status:** Past Due                                                                                 │
│  - **Customer Since:** August 22, 2022                                                                          │
│                                                                                                                 │
│  **Monthly Pro Plan Fee:**                                                                                      │
│  - In EUR: €261.68                                                                                              │
│  - In GBP: £222.14                                                                                              │
│                                                                                                                 │
│  **Recommended Action:** It is advised to reach out to Nova Retail Group to resolve the past due account        │
│  status and confirm the payment of the monthly Pro plan fee of €261.68 or £222.14, depending on their           │
│  preferred billing currency.                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---
## 13. Advanced: Human-in-the-Loop

In [ ]:
from crewai import Agent, Task, Crew, Process

# When human_input=True on a task, CrewAI will pause execution
# after the agent produces its draft output and prompt the user
# to review and optionally provide correction before finalizing.

legal_drafter = Agent(
    role="Legal Contract Drafter",
    goal="Draft legally sound contract clauses based on provided parameters.",
    backstory=(
        "You are a contract attorney specializing in SaaS and technology agreements. "
        "Your clauses are clear, enforceable, and client-protective."
    ),
    llm=llm,
    verbose=True,
)

draft_task = Task(
    description=(
        "Draft a limitation of liability clause for a SaaS agreement where: "
        "- Liability cap is 12 months of fees paid"
        "- Excludes gross negligence and willful misconduct from the cap"
        "- Mutual limitation (applies to both vendor and customer)"
        "- Governed by Delaware law"
    ),
    expected_output="A formal, numbered legal clause ready for insertion into a contract.",
    agent=legal_drafter,
    human_input=True,   # Agent pauses here for human review before finalizing
)

# Note: Running this cell will pause execution and prompt for input
# Uncomment to run:
legal_crew = Crew(
    agents=[legal_drafter],
    tasks=[draft_task],
    process=Process.sequential,
    verbose=True,
)
result = await legal_crew.kickoff_async()
print(result)

print("Human-in-the-loop task configured. Uncomment the crew kickoff to run interactively.")

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 9c919c34-89e2-49c2-8bd1-7e58bff09026                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Draft a limitation of liability clause for a SaaS agreement where: - Liability cap is 12 months of fees  │
│  paid- Excludes gross negligence and willful misconduct from the cap- Mutual limitation (applies to both        │
│  vendor and customer)- Governed by Delaware law                                                                 │
│  ID: 1594906b-9ddf-44a1-9333-5b809c47788b                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Legal Contract Drafter                                                                                  │
│                                                                                                                 │
│  Task: Draft a limitation of liability clause for a SaaS agreement where: - Liability cap is 12 months of fees  │
│  paid- Excludes gross negligence and willful misconduct from the cap- Mutual limitation (applies to both        │
│  vendor and customer)- Governed by Delaware law                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Legal Contract Drafter                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Limitation of Liability**                                                                                    │
│                                                                                                                 │
│  1. **Limitation of Liability**. Notwithstanding anything to the contrary in this Agreement, the maximum        │
│  aggregate liability of either Party to the other Party for any claims, damages, or losses arising out of or    │
│  related to this Agreement, whether in contract, tort (including negligence), or otherwise, shall not exceed    │
│  the total amount of fees paid by Customer to Vendor under this Agreement during the twelve (12) month period   │
│  immediately preceding the event giving rise to the claim.                                                      │
│                                                                                                                 │
│  2. **Exclusions**. The limitations set forth in Section 1 shall not apply to claims arising from (a) gross     │
│  negligence or willful misconduct of either Party; (b) breach of confidentiality obligations; or (c)            │
│  indemnification obligations as provided in this Agreement.                                                     │
│                                                                                                                 │
│  3. **Mutual Limitation**. The limitations of liability set forth in this Section shall apply equally to both   │
│  Parties and their respective officers, directors, employees, agents, and affiliates.                           │
│                                                                                                                 │
│  4. **Governing Law**. This limitation of liability shall be governed by and construed in accordance with the   │
│  laws of the State of Delaware, without regard to its conflict of laws principles.                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Draft a limitation of liability clause for a SaaS agreement where: - Liability cap is 12 months of fees  │
│  paid- Excludes gross negligence and willful misconduct from the cap- Mutual limitation (applies to both        │
│  vendor and customer)- Governed by Delaware law                                                                 │
│  Agent: Legal Contract Drafter                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

/tmp/ipykernel_1221/1193862473.py:39: RuntimeWarning: coroutine 'Crew.kickoff_async' was never awaited
  result = await legal_crew.kickoff_async()


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 9c919c34-89e2-49c2-8bd1-7e58bff09026                                                                       │
│  Final Output: **Limitation of Liability**                                                                      │
│                                                                                                                 │
│  1. **Limitation of Liability**: Except for liability arising from Gross Negligence or Willful Misconduct, in   │
│  no event shall either Party’s total aggregate liability to the other Party under this Agreement exceed the     │
│  total fees paid by the Customer for the Service during the twelve (12) months immediately preceding the event  │
│  giving rise to the liability. This limitation of liability shall apply to all claims arising out of or         │
│  related to this Agreement, whether based in contract, tort (including negligence), strict liability, or any    │
│  other theory of liability, and shall apply even if the Party has been advised of the possibility of such       │
│  damages.                                                                                                       │
│                                                                                                                 │
│  2. **Mutual Application**: This limitation of liability shall apply mutually to both the Vendor and the        │
│  Customer, ensuring that neither Party shall have greater liability than as specified herein.                   │
│                                                                                                                 │
│  3. **Governing Law**: This Limitation of Liability clause shall be governed by and construed in accordance     │
│  with the laws of the State of Delaware, without regard to its conflict of law principles.                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

**Limitation of Liability**

1. **Limitation of Liability**: Except for liability arising from Gross Negligence or Willful Misconduct, in no event shall either Party’s total aggregate liability to the other Party under this Agreement exceed the total fees paid by the Customer for the Service during the twelve (12) months immediately preceding the event giving rise to the liability. This limitation of liability shall apply to all claims arising out of or related to this Agreement, whether based in contract, tort (including negligence), strict liability, or any other theory of liability, and shall apply even if the Party has been advised of the possibility of such damages. 

2. **Mutual Application**: This limitation of liability shall apply mutually to both the Vendor and the Customer, ensuring that neither Party shall have greater liability than as specified herein.

3. **Governing Law**: This Limitation of Liability clause shall be governed by and construed in accordance with the laws



╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                                            │
│                                                                              │
│  Your preference has been saved. Future Crew/Flow executions will not        │
│  collect traces.                                                             │
│                                                                              │
│  To enable tracing later, do any one of these:                               │
│  • Set tracing=True in your Crew/Flow code                                   │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file               │
│  • Run: crewai traces enable                                                 │
│                                                                              │
╰─────────────────────────

---
## 14. Advanced: Async Execution and Callbacks

In [ ]:
from crewai import Agent, Task, Crew, Process
from datetime import datetime

# --- Step Callback: Called after every agent action ---
def step_callback(agent_action):
    timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[{timestamp}] STEP: {agent_action}")

# --- Task Callback: Called when a task completes ---
def task_callback(task_output):
    timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[{timestamp}] TASK COMPLETED. Output length: {len(str(task_output))} chars")

# --- Async Kickoff: Non-blocking execution ---
async def run_crew_async():
    summarizer = Agent(
        role="Executive Summarizer",
        goal="Produce concise executive summaries of lengthy documents.",
        backstory="You distill complex content into key decisions and action items.",
        llm=llm,
    )

    summary_task = Task(
        description=(
            "Summarize the following meeting notes into 5 bullet points, "
            "each focused on an action item with an owner and deadline:\n\n"
            "Meeting: Q3 Product Roadmap Review\n"
            "Sarah: We need to ship the new dashboard by end of October or we lose the Acme contract.\n"
            "Tom: The API team is blocked on authentication. They need a decision on OAuth vs SAML by Friday.\n"
            "Sarah: Marketing wants a demo environment set up by September 15 for the conference.\n"
            "James: We agreed to deprecate v1 API on November 1. We need a migration guide published by October 1.\n"
            "Tom: Budget approval for the new infrastructure is needed from Finance before September 30."
        ),
        expected_output="5 action items, each with: task, owner, and deadline.",
        agent=summarizer,
        callback=task_callback,
    )

    crew = Crew(
        agents=[summarizer],
        tasks=[summary_task],
        process=Process.sequential,
        step_callback=step_callback,
        verbose=True,
    )

    # kickoff_async returns a coroutine; await it in an async context
    result = await crew.kickoff_async()
    return result

# In a Jupyter notebook, use await directly
import asyncio
result = await run_crew_async()
print("\nFINAL OUTPUT:")
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 82af5553-61f2-4630-9b0f-5b9a07f80fa4                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Summarize the following meeting notes into 5 bullet points, each focused on an action item with an       │
│  owner and deadline:                                                                                            │
│                                                                                                                 │
│  Meeting: Q3 Product Roadmap Review                                                                             │
│  Sarah: We need to ship the new dashboard by end of October or we lose the Acme contract.                       │
│  Tom: The API team is blocked on authentication. They need a decision on OAuth vs SAML by Friday.               │
│  Sarah: Marketing wants a demo environment set up by September 15 for the conference.                           │
│  James: We agreed to deprecate v1 API on November 1. We need a migration guide published by October 1.          │
│  Tom: Budget approval for the new infrastructure is needed from Finance before September 30.                    │
│  ID: 79a1a20c-28c9-4351-8a68-d9768805be45                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Executive Summarizer                                                                                    │
│                                                                                                                 │
│  Task: Summarize the following meeting notes into 5 bullet points, each focused on an action item with an       │
│  owner and deadline:                                                                                            │
│                                                                                                                 │
│  Meeting: Q3 Product Roadmap Review                                                                             │
│  Sarah: We need to ship the new dashboard by end of October or we lose the Acme contract.                       │
│  Tom: The API team is blocked on authentication. They need a decision on OAuth vs SAML by Friday.               │
│  Sarah: Marketing wants a demo environment set up by September 15 for the conference.                           │
│  James: We agreed to deprecate v1 API on November 1. We need a migration guide published by October 1.          │
│  Tom: Budget approval for the new infrastructure is needed from Finance before September 30.                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Executive Summarizer                                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  - **Task:** Ship the new dashboard                                                                             │
│    **Owner:** Sarah                                                                                             │
│    **Deadline:** End of October                                                                                 │
│                                                                                                                 │
│  - **Task:** Decide on authentication method (OAuth vs SAML)                                                    │
│    **Owner:** Tom                                                                                               │
│    **Deadline:** Friday                                                                                         │
│                                                                                                                 │
│  - **Task:** Set up demo environment for marketing                                                              │
│    **Owner:** Sarah                                                                                             │
│    **Deadline:** September 15                                                                                   │
│                                                                                                                 │
│  - **Task:** Publish migration guide for deprecating v1 API                                                     │
│    **Owner:** James                                                                                             │
│    **Deadline:** October 1                                                                                      │
│                                                                                                                 │
│  - **Task:** Obtain budget approval for new infrastructure                                                      │
│    **Owner:** Tom                                                                                               │
│    **Deadline:** Before September 30                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[10:03:31] TASK COMPLETED. Output length: 531 chars


╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Summarize the following meeting notes into 5 bullet points, each focused on an action item with an       │
│  owner and deadline:                                                                                            │
│                                                                                                                 │
│  Meeting: Q3 Product Roadmap Review                                                                             │
│  Sarah: We need to ship the new dashboard by end of October or we lose the Acme contract.                       │
│  Tom: The API team is blocked on authentication. They need a decision on OAuth vs SAML by Friday.               │
│  Sarah: Marketing wants a demo environment set up by September 15 for the conference.                           │
│  James: We agreed to deprecate v1 API on November 1. We need a migration guide published by October 1.          │
│  Tom: Budget approval for the new infrastructure is needed from Finance before September 30.                    │
│  Agent: Executive Summarizer                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FINAL OUTPUT:
- **Task:** Ship the new dashboard  
  **Owner:** Sarah  
  **Deadline:** End of October  

- **Task:** Decide on authentication method (OAuth vs SAML)  
  **Owner:** Tom  
  **Deadline:** Friday  

- **Task:** Set up demo environment for marketing  
  **Owner:** Sarah  
  **Deadline:** September 15  

- **Task:** Publish migration guide for deprecating v1 API  
  **Owner:** James  
  **Deadline:** October 1  

- **Task:** Obtain budget approval for new infrastructure  
  **Owner:** Tom  
  **Deadline:** Before September 30  


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 82af5553-61f2-4630-9b0f-5b9a07f80fa4                                                                       │
│  Final Output: - **Task:** Ship the new dashboard                                                               │
│    **Owner:** Sarah                                                                                             │
│    **Deadline:** End of October                                                                                 │
│                                                                                                                 │
│  - **Task:** Decide on authentication method (OAuth vs SAML)                                                    │
│    **Owner:** Tom                                                                                               │
│    **Deadline:** Friday                                                                                         │
│                                                                                                                 │
│  - **Task:** Set up demo environment for marketing                                                              │
│    **Owner:** Sarah                                                                                             │
│    **Deadline:** September 15                                                                                   │
│                                                                                                                 │
│  - **Task:** Publish migration guide for deprecating v1 API                                                     │
│    **Owner:** James                                                                                             │
│    **Deadline:** October 1                                                                                      │
│                                                                                                                 │
│  - **Task:** Obtain budget approval for new infrastructure                                                      │
│    **Owner:** Tom                                                                                               │
│    **Deadline:** Before September 30                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---
## 15. Best Practices and Common Pitfalls

### Best Practices

**Agent Design**
- Give each agent a single, clear responsibility. An agent trying to do too many things produces mediocre output for each.
- Write backstories that are specific and credible. Vague backstories produce generic responses.
- Set `allow_delegation=False` on specialized agents to prevent uncontrolled delegation chains.

**Task Design**
- Use numbered requirements in task descriptions. Agents are more likely to cover every point.
- The `expected_output` field acts as a quality rubric — the more specific, the better the output.
- Use `output_pydantic` when downstream code needs to consume structured data.

**Crew Design**
- Default to `Process.sequential` unless you need dynamic orchestration.
- Enable `memory=True` with `text-embedding-3-small` when agents need to reference earlier context.
- Use `max_rpm` at the crew level to avoid OpenAI rate limit errors in large crews.

### Common Pitfalls

| Pitfall | Cause | Fix |
|---|---|---|
| Agent produces off-topic output | Backstory or goal too vague | Make backstory more specific and role-constraining |
| Infinite tool loops | `max_iter` too high | Set `max_iter=8` or lower for most tasks |
| Context not flowing between tasks | Missing `context=[prev_task]` | Explicitly link dependent tasks via `context` |
| Pydantic validation errors | LLM output format inconsistent | Add format examples to `expected_output` |
| Rate limit errors on large crews | Too many simultaneous API calls | Set `max_rpm` at agent and crew level |